# Data Exploration
Read Parquet files from MinIO, explore with pandas.

In [ ]:
import boto3
import os
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('MINIO_ENDPOINT', 'http://localhost:9000'),
    aws_access_key_id=os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    aws_secret_access_key=os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
print('S3 client ready:', s3.meta.endpoint_url)

In [ ]:
bucket = 'raw-data'
response = s3.list_objects_v2(Bucket=bucket)
objects = [obj['Key'] for obj in response.get('Contents', [])]
parquet_files = [k for k in objects if k.endswith('.parquet')]
print(f'Found {len(parquet_files)} Parquet file(s) in {bucket}:')
for f in parquet_files:
    print(' -', f)

In [ ]:
import pyarrow.fs as pafs
import pyarrow.parquet as pq

endpoint = os.getenv('MINIO_ENDPOINT', 'http://localhost:9000').replace('http://', '').replace('https://', '')
use_ssl = os.getenv('MINIO_ENDPOINT', 'http://localhost:9000').startswith('https')

fs = pafs.S3FileSystem(
    endpoint_override=endpoint,
    access_key=os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    secret_key=os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    scheme='https' if use_ssl else 'http'
)

first_file = parquet_files[0] if parquet_files else None
if first_file:
    table = pq.read_table(f'{bucket}/{first_file}', filesystem=fs)
    print(f'Read {table.num_rows} rows, {table.num_columns} columns from {first_file}')
else:
    print('No Parquet files found. Upload data to the raw-data bucket first.')

In [ ]:
import pandas as pd

if first_file:
    df = table.to_pandas()
    print(df.shape)
    display(df.describe())
else:
    # Demo with synthetic data
    import numpy as np
    df = pd.DataFrame({
        'order_id': range(1000),
        'amount': np.random.exponential(50, 1000),
        'category': np.random.choice(['electronics', 'clothing', 'food'], 1000),
        'region': np.random.choice(['north', 'south', 'east', 'west'], 1000)
    })
    print('Using synthetic demo data')
    display(df.describe())

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

In [ ]:
import matplotlib.pyplot as plt

num_cols = df.select_dtypes(include='number').columns.tolist()
fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 4))
if len(num_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols):
    df[col].hist(bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
import io
import pyarrow as pa

out_bucket = 'processed-data'
out_key = 'explored/output.parquet'

# Ensure bucket exists
existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if out_bucket not in existing:
    s3.create_bucket(Bucket=out_bucket)

buf = io.BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=out_bucket, Key=out_key, Body=buf.read())
print(f'Written processed data to s3://{out_bucket}/{out_key}')